# Step 9b · round-2 adversarial training + registry-relative calibration (DeBERTa)
Same as kaggle_cal_adv, but the ADV twin is trained with the round-2 data (15% + 30% white-box rewrites of train seeds,
1,400 total). ~12 min. Note: the wb30 sets are now partly training data (train seeds); read the held-out rows.
**Settings:** GPU **T4 x1** · Internet **On** · Secrets **Hugging Face** + **mcp-git** · Save Version → Save & Run All.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
GPU_DEPS = False
MIN_COMMIT_FILE = "scripts/calibrate_adv.py"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/calibrate_adv.py --tag cal_adv2_deb --set 'caladv.adv_sources=[results/wb1,results/wb1e5,results/wb3_base_b30_deb,results/wb3_adv_b30_deb]'


In [ ]:
import glob
for run in sorted(glob.glob("/kaggle/working/outputs/calibrate_adv/*")):
    print(open(f"{run}/cal_adv.md").read())
